## Imports

In [1]:
import os

import numpy as np
import tflite
from tensorflow import lite as interpreter_wrapper
# import tensorflow.compat.v1 as tf
# import tensorflow as tf

import tvm
import tvm.relay.testing.tf as tf_testing
from tvm.contrib.download import download_testdata
from tvm import relax, relay
from tvm.contrib import graph_executor
from tvm.relax.frontend.tflite import from_tflite

2024-05-16 10:09:14.837784: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-05-16 10:09:14.886992: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-05-16 10:09:15.702399: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
[10:09:16] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled with support for Arm(R)-based targets.
[10:09:16] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warn

## Models

### Mobilenet V2 (fp32)

In [7]:
def mobilenet_v2_fp32():
    tflite_model_file = tf_testing.get_workload_official(
        "http://download.tensorflow.org/models/tflite_11_05_08/mobilenet_v2_1.0_224.tgz",
        "mobilenet_v2_1.0_224.tflite",
    )
    with open(tflite_model_file, "rb") as f:
        tflite_model_buf = f.read()
    shape = (1, 224, 224, 3)
    data = np.random.uniform(size=shape).astype("float32")
    tflite_model = tflite.Model.GetRootAsModel(tflite_model_buf, 0)
    
    shape_dict = {"input": shape}
    dtype_dict = {"input": "float32"}
    input_info = [(shape_dict[name], dtype_dict[name]) for name in shape_dict]
    input_node = ["input"]
    return tflite_model, tflite_model_buf, input_info, input_node, shape_dict, dtype_dict, data

### Mobilenet V2 (int8)

In [8]:
def mobilenet_v2_int8():
    tflite_model_file = download_testdata(
        "https://raw.githubusercontent.com/dmlc/web-data/main/tensorflow/models/Quantized/"
        "mobilenet_v2_quantized.tflite",
        "mobilenet_v2_quantized.tflite",
    )
    with open(tflite_model_file, "rb") as f:
        tflite_model_buf = f.read()
    shape = (1, 224, 224, 3)
    data = np.random.uniform(size=shape).astype("float32")
    tflite_model = tflite.Model.GetRootAsModel(tflite_model_buf, 0)

    shape_dict = {"input": shape}
    dtype_dict = {"input": "float32"}  # I/O is still fp32!
    input_info = [(shape_dict[name], dtype_dict[name]) for name in shape_dict]
    input_node = ["input_1"]
    return tflite_model, tflite_model_buf, input_info, input_node, shape_dict, dtype_dict, data

### Select Model

In [9]:
# tflite_model, tflite_model_buf, input_info, input_node, shape_dict, dtype_dict, data = mobilenet_v2_fp32()
tflite_model, tflite_model_buf, input_info, input_node, shape_dict, dtype_dict, data = mobilenet_v2_int8()

## Convert

### Relax

Run frontend

In [10]:
relax_mod = from_tflite(tflite_model, shape_dict=shape_dict, dtype_dict=dtype_dict)

[10:10:13] /var/tmp/ga87puy/tvm_relax/src/relax/op/tensor/qdq.cc:82: input_sinfo->ndim=4

[10:10:13] /var/tmp/ga87puy/tvm_relax/src/relax/op/tensor/qdq.cc:83: attrs->axis=-1

[10:10:13] /var/tmp/ga87puy/tvm_relax/src/relax/op/tensor/qdq.cc:85: axis=3

[10:10:13] /var/tmp/ga87puy/tvm_relax/src/relax/op/tensor/qdq.cc:86: (axis < 0)=0

[10:10:13] /var/tmp/ga87puy/tvm_relax/src/relax/op/tensor/qdq.cc:87: (axis > input_sinfo->ndim - 1)=0

[10:10:13] /var/tmp/ga87puy/tvm_relax/src/relax/op/tensor/qdq.cc:169: input_sinfo->ndim=4

[10:10:13] /var/tmp/ga87puy/tvm_relax/src/relax/op/tensor/qdq.cc:170: attrs->axis=-1

[10:10:13] /var/tmp/ga87puy/tvm_relax/src/relax/op/tensor/qdq.cc:176: axis=0

[10:10:13] /var/tmp/ga87puy/tvm_relax/src/relax/op/tensor/qdq.cc:169: input_sinfo->ndim=0

[10:10:13] /var/tmp/ga87puy/tvm_relax/src/relax/op/tensor/qdq.cc:170: attrs->axis=-1

[10:10:13] /var/tmp/ga87puy/tvm_relax/src/relax/op/tensor/qdq.cc:176: axis=0

[10:10:13] /var/tmp/ga87puy/tvm_relax/src/relax/op/t

### Relay

In [11]:
relay_mod, relay_params = relay.frontend.from_tflite(tflite_model, shape_dict=shape_dict, dtype_dict=dtype_dict)

## Display

### Relax

In [12]:
relax_mod.show()

### Relay

In [13]:
relay_mod.show()

## Build

In [14]:
TARGET = "llvm"
target = tvm.target.Target(TARGET, host=TARGET)

### Relax

In [15]:
with tvm.transform.PassContext(opt_level=3):
    relax_lib = relax.build(relax_mod, target)

### Relay

In [16]:
with tvm.transform.PassContext(opt_level=3):
    relay_lib = relay.build(relay_mod, target, params=relay_params)

conv2d NHWC layout is not optimized for x86 with autotvm.
depthwise_conv2d NHWC layout is not optimized for x86 with autotvm.
conv2d NHWC layout is not optimized for x86 with autotvm.
conv2d NHWC layout is not optimized for x86 with autotvm.
depthwise_conv2d NHWC layout is not optimized for x86 with autotvm.
conv2d NHWC layout is not optimized for x86 with autotvm.
conv2d NHWC layout is not optimized for x86 with autotvm.
depthwise_conv2d NHWC layout is not optimized for x86 with autotvm.
conv2d NHWC layout is not optimized for x86 with autotvm.
conv2d NHWC layout is not optimized for x86 with autotvm.
depthwise_conv2d NHWC layout is not optimized for x86 with autotvm.
conv2d NHWC layout is not optimized for x86 with autotvm.
conv2d NHWC layout is not optimized for x86 with autotvm.
depthwise_conv2d NHWC layout is not optimized for x86 with autotvm.
conv2d NHWC layout is not optimized for x86 with autotvm.
conv2d NHWC layout is not optimized for x86 with autotvm.
depthwise_conv2d NHWC 

## Run

### Relax (VM)

In [17]:
def run_relax(relax_lib, input_data):
    input_data = [input_data]
    vm = relax.VirtualMachine(relax_lib, tvm.cpu())
    vm.set_input("main", *input_data)
    vm.invoke_stateful("main")
    tvm_output = vm.get_outputs("main")
    tvm_output = np.expand_dims(tvm_output.numpy(), 0)
    return tvm_output

In [18]:
relax_output = run_relax(relax_lib, data)

In [19]:
# relax_output
# relax_output.shape

### Relay

In [20]:
def run_relay(relay_lib, input_data, input_node, num_output=1):
    dev = tvm.device(TARGET, 0)
    m = graph_executor.GraphModule(relay_lib["default"](dev))
    for i, node in enumerate(input_node):
        m.set_input(node, tvm.nd.array(input_data[i].astype(input_data[i].dtype)))
    m.run()
    tvm_output_list = []
    for i in range(0, num_output):
        tvm_output = m.get_output(i)
        tvm_output_list.append(tvm_output.numpy())
    return np.array(tvm_output_list)

In [21]:
relay_output = run_relay(relay_lib, data, input_node)

In [22]:
# relay_output
# relay_output.shape

### TFLite Interpreter

In [23]:
def run_tflite(tflite_model_buf, input_data):
    input_data = [input_data]
    interpreter = interpreter_wrapper.Interpreter(model_content=tflite_model_buf)
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    # for i, input_detail in enumerate(input_details):
    #    interpreter.resize_tensor_input(input_detail["index"], input_data[i].shape)
    interpreter.allocate_tensors()
    assert len(input_data) == len(input_details)
    for i, input_detail in enumerate(input_details):
        interpreter.set_tensor(input_detail["index"], input_data[i])
    interpreter.invoke()
    tflite_output = []
    for _, output_detail in enumerate(output_details):
        tflite_output.append(interpreter.get_tensor(output_detail["index"]))
    return np.array(tflite_output)

In [24]:
tflite_output = run_tflite(tflite_model_buf, data)

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [25]:
# tflite_output
# tflite_output.shape

## Compare

### Relax vs. Relay

In [26]:
tvm.testing.assert_allclose(relax_output, relay_output, rtol=1e-4, atol=1e-4)

AssertionError: 
Not equal to tolerance rtol=0.0001, atol=0.0001

Mismatched elements: 972 / 1001 (97.1%)
Max absolute difference: 9.958812
Max relative difference: 8.
 x: array([[[-0.306425,  0.      , -0.229819, ..., -0.076606, -0.153212,
         -0.153212]]], dtype=float32)
 y: array([[[ 0.153212,  0.229819,  1.608731, ...,  0.459637, -0.766062,
          1.149094]]], dtype=float32)

### Relax vs. TFLite

In [27]:
tvm.testing.assert_allclose(relax_output, tflite_output, rtol=1e-4, atol=1e-4)

AssertionError: 
Not equal to tolerance rtol=0.0001, atol=0.0001

Mismatched elements: 980 / 1001 (97.9%)
Max absolute difference: 10.188631
Max relative difference: 9.
 x: array([[[-0.306425,  0.      , -0.229819, ..., -0.076606, -0.153212,
         -0.153212]]], dtype=float32)
 y: array([[[ 0.153212,  0.459637,  1.532125, ...,  0.459637, -0.766062,
          1.2257  ]]], dtype=float32)

### Relay vs. TFLite

In [28]:
tvm.testing.assert_allclose(relay_output, tflite_output, rtol=1e-4, atol=1e-4)

AssertionError: 
Not equal to tolerance rtol=0.0001, atol=0.0001

Mismatched elements: 750 / 1001 (74.9%)
Max absolute difference: 0.45963764
Max relative difference: 4.
 x: array([[[ 0.153212,  0.229819,  1.608731, ...,  0.459637, -0.766062,
          1.149094]]], dtype=float32)
 y: array([[[ 0.153212,  0.459637,  1.532125, ...,  0.459637, -0.766062,
          1.2257  ]]], dtype=float32)

In [35]:
tvm.testing.assert_allclose(relay_output, tflite_output, rtol=0.3, atol=0.3)

AssertionError: 
Not equal to tolerance rtol=0.3, atol=0.3

Mismatched elements: 3 / 1001 (0.3%)
Max absolute difference: 0.45963764
Max relative difference: 4.
 x: array([[[ 0.153212,  0.229819,  1.608731, ...,  0.459637, -0.766062,
          1.149094]]], dtype=float32)
 y: array([[[ 0.153212,  0.459637,  1.532125, ...,  0.459637, -0.766062,
          1.2257  ]]], dtype=float32)